# mask + prepare images for AutoProf + run Autoprof

> Optionally mask sources in a Euclid or Mock image prior to ICL measurements.

> Set NaN values to finite value (default 99), as cleaned images, to prep for autoprof

> Run Autoprof on images, using the measurement mask from step one.

> Extract Surface Brightness Profile with measuring median within annuli, and compare to Autoprof resutls.

In [1]:
# | default_exp euclid.mask

In [37]:
# | exporti
import pandas as pd 

import glob
from pathlib import Path
import numpy as np

import astropy.units as u
from astropy.io import fits
from astropy.wcs import WCS
from astropy.visualization import simple_norm
from astropy.stats import sigma_clip
from astropy.coordinates import SkyCoord
from astropy.nddata import CCDData

from nicl.mask import (
    create_bcg_mask,
    create_icl_mask,
    create_faint_mask,
    create_object_mask,
)
from nicl.utilities import (
    calc_sb_threshold as _calc_sb_threshold,
    get_pixel_scale,
    sb_to_adu,
)
from nicl.background import get_background, plot_background


In [38]:
# | hide

import logging

import matplotlib.pyplot as plt


from nicl.main import configure_logging
from nicl.mask import plot_mask

import importlib.util
import sys
import os
import traceback
from textwrap import dedent

import autoprof
from autoprof import Pipeline 

from photutils.aperture import EllipticalAnnulus, CircularAnnulus
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse, Circle
from scipy.stats import median_abs_deviation

from astropy.cosmology import Planck18 as cosmo
from astropy.cosmology import FlatLambdaCDM
h=0.7
cosmo=FlatLambdaCDM(H0=h*100, Om0=0.3)

In [4]:
# | exporti
plt.style.use("nicl.euclid.v1nicl")

### masking is too sensitive to NSIGMA in icl mask


In [5]:
######### Parameters below are being called in create_masks
ICL_NSIGMA = 0.5    
ICL_SMOOTH_SIGMA = 50.0
ICL_BKG_BOX_SIZE = 300
ICL_BKG_FILTER_SIZE = 3
ICL_DILATION_RADIUS = 70.0
ICL_MEDIAN_FILTER = True
REGULAR_DETECTION_PARAMS = {
    "nsigma": 3.0,
    "background": 0.0,
    "smooth_sigma": 1.0,
    "npixels": 5,
    "nlevels": 32,
    "contrast": 0.01,
    "bkg_box_size": 200,
    "bkg_filter_size": 3,
}
REGULAR_GROWTH = 0.5
FAINT_DETECTION_PARAMS = {
    "nsigma": 2.0,
    "smooth_sigma": 1.0,
    "npixels": 5,
    "nlevels": 128,
    "contrast": 0.001,
}
FAINT_GROWTH = 0.25
FAINT_BKG_SIGMA = 50.0
NIR_STACK_BKG_BOX_SIZE = 300
NIR_STACK_BKG_FILTER_SIZE = 3

In [6]:

def calc_sb_threshold(
    z,  # cluster redshift
    filter,  # Euclid filter: VIS, Y, J or H
):
    """Determine the ICL surface brightness threshold for a given redshift and Euclid filter."""
    filter = filter.replace("NIR_", "")
    filter = f"Euclid-{filter}.ecsv"
    return _calc_sb_threshold(z, filter)
    
def create_masks(
    image,  # the image to mask, with bad pixels set to NaN, as a CCDImage or filename
    *,  # the following parameters must be provided as keyword arguments if required
    z=None,  # the cluster redshift for the BCG mask; if None then returned BCG mask is None
    filter=None,  # the filter name for the BCG mask; if None then returned BCG mask is None
    centre_pos=None,  # the position of the BCG/cluster centre; set to False for a non-cluster image
    make_faint_mask=True,  # whether to create a separate object mask in the ICL region
    zeropoint= "ZPAB",  # the zeropoint, either as a header keyword or numeric value
):
    """Create BCG, ICL, object and faint masks with default settings for Euclid data.

    These default settings are to be refined.
    """
    if isinstance(image, str):
        image = CCDData.read(image, unit="adu")  # unit does not matter
    # === Bad pixels ===
    badpixel_mask = ~np.isfinite(image)
    # === BCG ===
    if centre_pos is not False and z is not None and filter is not None:
        sb_threshold = calc_sb_threshold(z, filter)
        if isinstance(zeropoint, str):
            zp = image.header[zeropoint] * u.ABmag
        else:
            zp = zeropoint * u.ABmag
        sb_adu_threshold = sb_to_adu(sb_threshold, get_pixel_scale(image), zp)
        bcg_mask = create_bcg_mask(
            image.data,
            sb_threshold=sb_adu_threshold,
            centre_pos=centre_pos,
            wcs=image.wcs,
        )
    else:
        bcg_mask = None
    # === ICL ===
    if centre_pos is not False:
        icl_mask, _ = create_icl_mask(
            image.data,
            centre_pos=centre_pos,
            wcs=image.wcs,
            nsigma=ICL_NSIGMA,
            smooth_sigma=ICL_SMOOTH_SIGMA,
            bkg_box_size=ICL_BKG_BOX_SIZE,
            bkg_filter_size=ICL_BKG_FILTER_SIZE,
            dilation_radius=ICL_DILATION_RADIUS,
            median_filter=ICL_MEDIAN_FILTER,
        )
    else:
        icl_mask = None
    # === Regular objects ===
    if make_faint_mask:
        # In this case, we exclude objects under the ICL mask,
        # as these will be included in the faint mask
        object_mask, bkg, threshold, centre_mask = create_object_mask(
            image.data,
            exclude_mask=icl_mask,
            growth=REGULAR_GROWTH,
            detection_params=REGULAR_DETECTION_PARAMS,
        )
    else:
        # In this case, we only exclude the object at the central position
        object_mask, bkg, threshold, centre_mask = create_object_mask(
            image.data,
            exclude_position=centre_pos,
            wcs=image.wcs,
            growth=REGULAR_GROWTH,
            detection_params=REGULAR_DETECTION_PARAMS,
        )
    # === Faint objects (under ICL) ===
    if centre_pos is not False and make_faint_mask:
        faint_mask, faint_bkg, faint_threshold = create_faint_mask(
            image.data,
            include_mask=centre_mask,
            exclude_position=centre_pos,
            wcs=image.wcs,
            growth=FAINT_GROWTH,
            detection_params=FAINT_DETECTION_PARAMS,
            bkg_sigma=FAINT_BKG_SIGMA,
        )
    else:
        faint_mask = None
    output_masks = {
        "badpixel": badpixel_mask,
        "bcg": bcg_mask,
        "icl": icl_mask,
        "object": object_mask,
        "faint": faint_mask,
    }
    return output_masks

# updated to include wcs information while storing masks
def save_masks(
    masks,    # Dictionary of masks returned by create_masks
    output_dir,  
    label=None,   
    reference_header=None 
):
    """Save all masks produced by create_masks to disk."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    prefix = f"{label}_" if label else ""
    for mask_name, mask in masks.items():
        if mask is not None:
            mask = mask.astype(np.uint8)
            output_fn = output_dir / f"{prefix}{mask_name}_mask.fits"
            hdu = fits.PrimaryHDU(data=mask, header=reference_header)
            hdu.writeto(output_fn, overwrite=True)
            print(f"Saved mask: {output_fn}")


In [49]:
# | export

### updated nir_mask function, changed get_fitsdata to CCDDATA because we require the image.wcs in masksking procedures. The masking will place bkgsub images to a temp folder and use within stacking code. Then will get rid of them
### additionally, included centre_pos parameter to both nir and vis masks to be able to create a combined nir mask for skypatch.

def stack_nir_bands(H_filename, J_filename, Y_filename, output_dir=None, label=None):
    """Stack NIR images and optionally save to disk. Returns a CCDData object."""
    logger = logging.getLogger(__name__)
    logger.info(f"Stacking images: {H_filename}, {J_filename}, {Y_filename}")
    
    # Reading images as CCDData to preserve WCS
    H = CCDData.read(H_filename, unit="adu")
    J = CCDData.read(J_filename, unit="adu")
    Y = CCDData.read(Y_filename, unit="adu")

    images = {"H": H.data, "J": J.data, "Y": Y.data}
    masks = {band: ~np.isfinite(images[band]) for band in ["H", "J", "Y"]}
    global_mask = masks["H"] | masks["J"] | masks["Y"]
    
    for band in ["H", "J", "Y"]:
        images[band][global_mask] = np.nan

    bandwidth = {"H": 499.9, "J": 399.4, "Y": 262.7}
    total_bandwidth = sum(bandwidth.values())
    
    combined_image = (
        (images["H"] * bandwidth["H"])
        + (images["J"] * bandwidth["J"])
        + (images["Y"] * bandwidth["Y"])
    ) / total_bandwidth

    logger.debug(
        f"Combined Image: Min={np.nanmin(combined_image)}, Max={np.nanmax(combined_image)}, NaN Count={np.isnan(combined_image).sum()}"
    )

    # Using H-band WCS as reference
    combined_ccd = CCDData(combined_image, unit="adu", wcs=H.wcs)

    if output_dir:
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)
        prefix = f"{label}_" if label else ""
        header = combined_ccd.wcs.to_header()
        fits.writeto(
            output_dir / f"{prefix}HJY_combined.fits",
            combined_ccd.data,
            header=header,
            overwrite=True,
        )

    return combined_ccd
    

def create_combined_nir_mask(
    H_filename, J_filename, Y_filename, output_dir=None, label=None, centre_pos=None, redshift=None, filter=None, zeropoint=23.9
):
    """Create a combined NIR mask for use when measuring ICL."""

    filenames = {"H": H_filename, "J": J_filename, "Y": Y_filename}
    images = {}

    for band in ["H", "J", "Y"]:
        image = CCDData.read(filenames[band], unit="adu")
        
        m = create_masks(image, make_faint_mask=False, centre_pos=centre_pos, z=redshift, filter=filter, zeropoint=23.9)

        if centre_pos is not False:
            mask_for_background = m["badpixel"] | m["icl"] | m["object"]
            
        else:
            mask_for_background = m["badpixel"] | m["object"]

        bkg = get_background(
            image.data,
            mask=mask_for_background,
            box_size=NIR_STACK_BKG_BOX_SIZE,
            filter_size=NIR_STACK_BKG_FILTER_SIZE,
        )

        image_bkg_sub = image.data - bkg.background
        images[band] = image_bkg_sub

        # Saving temp background-subtracted image
        temp_filename = f"_temp_bkgsub_{band}.fits"
        temp_ccd = CCDData(image_bkg_sub, unit="adu", wcs=image.wcs)
        temp_ccd.write(temp_filename, overwrite=True)
        filenames[band] = temp_filename

    # Now pass temp files to stacking function
    print("Proceeding to stack_nir_bands...")
    combined_ccd = stack_nir_bands(filenames["H"], filenames["J"], filenames["Y"], output_dir=None, label=None)
    print("stack_nir_bands is complete...")
    
    if output_dir:
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)
        prefix = f"{label}_" if label else ""
        header = combined_ccd.wcs.to_header()
        
        fits.writeto(
            output_dir / f"{prefix}NIR_YJH_COADDED.fits",
            combined_ccd.data,
            header=header,
            overwrite=True
        )
        print('Coadded image is saved.')
        
    print("Masking the coadded image...")
    masks = create_masks(combined_ccd, centre_pos=centre_pos, z=redshift, filter=filter, zeropoint=23.9)

    if centre_pos is not False:
        mask_for_background = masks["badpixel"] | masks["icl"] | masks["object"]
    else:
        mask_for_background = masks["badpixel"] | masks["object"]
        
    faint_mask = masks["faint"] if masks["faint"] is not None else np.zeros_like(masks["object"], dtype=bool)
    mask_for_measurement = masks["badpixel"] | masks["object"] | faint_mask
    
    print("Masks are generated...")
    
    if output_dir:
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)
        prefix = f"{label}_" if label else ""
        header = combined_ccd.wcs.to_header()

        fits.writeto(
            output_dir / f"{prefix}NIR_background_mask.fits",
            mask_for_background.astype("uint8"),
            header=header,
            overwrite=True,
        )
        fits.writeto(
            output_dir / f"{prefix}NIR_measurement_mask.fits",
            mask_for_measurement.astype("uint8"),
            header=header,
            overwrite=True,
        )
        

    # Cleanup temp files
    for band in ["H", "J", "Y"]:
        temp_file = filenames[band]
        if "_temp_bkgsub_" in temp_file and Path(temp_file).exists():
            Path(temp_file).unlink()

    return mask_for_background, mask_for_measurement, combined_ccd.data
    

def create_vis_mask(VIS_filename, output_dir=None, label=None, centre_pos=None, redshift=None, filter=None):
    """Create a VIS mask for use when measuring ICL."""
    
    image = CCDData.read(VIS_filename, unit="adu")
    masks = create_masks(image, centre_pos=centre_pos, z=redshift, filter=filter)

    if centre_pos is not False:
        mask_for_background = masks["badpixel"] | masks["icl"] | masks["object"]
    else:
        mask_for_background = masks["badpixel"] | masks["object"]
        
    faint_mask = masks["faint"] if masks["faint"] is not None else np.zeros_like(masks["object"], dtype=bool)
    mask_for_measurement = masks["badpixel"] | masks["object"] | faint_mask

    if output_dir:
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)
        prefix = f"{label}_" if label else ""
        header = image.wcs.to_header()

        fits.writeto(
            output_dir / f"{prefix}VIS_background_mask.fits",
            mask_for_background.astype("uint8"),
            header=header,
            overwrite=True,
        )
        fits.writeto(
            output_dir / f"{prefix}VIS_measurement_mask.fits",
            mask_for_measurement.astype("uint8"),
            header=header,
            overwrite=True,
        )

    return mask_for_measurement, mask_for_background


In [10]:
from photutils.aperture import CircularAnnulus
from photutils.utils import calc_total_error
from matplotlib.colors import Normalize
from scipy.stats import median_abs_deviation
from astropy.stats import sigma_clip
from astropy.visualization import simple_norm
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import glob
import numpy as np
import pandas as pd
from pathlib import Path
from astropy.io import fits
from astropy.nddata import CCDData


# New cleaning function...

def create_bkgsub_images(
    image_paths,
    measurement_mask_path,
    output_background_dir,
    temp_cleaned_dir,
    centre_pos=False,
    label= None,
    clean_nans=False
):
    """
    Loads images, applies the measurement mask to estimate background,
    saves the background maps, and writes cleaned background-subtracted images with NaNs set to 99.
    """
    Path(output_background_dir).mkdir(parents=True, exist_ok=True)
    Path(temp_cleaned_dir).mkdir(parents=True, exist_ok=True)

    measurement_mask = fits.getdata(measurement_mask_path).astype(bool)

    filenames_cleaned = {}

    for band, image_path in image_paths.items():
        image = CCDData.read(image_path, unit="adu")

        bkg = get_background(
            image.data,
            mask=measurement_mask,
            box_size=NIR_STACK_BKG_BOX_SIZE,
            filter_size=NIR_STACK_BKG_FILTER_SIZE,
        )

        # Save background
        bkg_filename = Path(output_background_dir) / f"{label}_bkg.fits"
        
        fits.writeto(
            bkg_filename,
            bkg.background,
            header=image.wcs.to_header(),
            overwrite=True
        )

        # Subtract background and clean NaNs
        image_bkg_sub = image.data - bkg.background

        if clean_nans:
            image_bkg_sub = np.where(np.isfinite(image_bkg_sub), image_bkg_sub, 99)
            cleaned_filename = Path(temp_cleaned_dir) / f"{label}_cleaned_bkgsub.fits"
            
        else:
            cleaned_filename = Path(temp_cleaned_dir) / f"{label}_bkgsub.fits"
        
        cleaned_ccd = CCDData(image_bkg_sub, unit="adu", wcs=image.wcs)
        cleaned_ccd.write(cleaned_filename, overwrite=True)

        filenames_cleaned[band] = str(cleaned_filename)

    return filenames_cleaned


In [11]:
# | export

def run_autoprof(
    ids,
    image_files,
    mask_files=None,
    mode="image list",
    unit_type="intensity",
    gscale=0.4,
    pixelscale=0.3,
    zeropoint=23.9,
    out_dir="./",
    config_name="basic_config.py",
    log_path="AutoProf.log",
    fourier_orders=None,
    forced_photometry=False,
    forced_profile_filter=None,
    forced_profile_path=None, # .prof file path "dir/x.prof"
):
    """
    Running AutoProf with optional fixed photometry.

    If forced_photometry=True, the reference profile in forced_profile_path is used,
    and the pipeline steps are adjusted accordingly.
    """

    os.makedirs(out_dir, exist_ok=True)
    config_file = os.path.join(out_dir, config_name)

    # Modify pipeline based on fixed photometry
    if forced_photometry:
        if not forced_profile_path:
            raise ValueError("forced_profile_path must be provided when forced_photometry=True.")
        if not forced_profile_filter:
            raise ValueError("forced_profile_filter must be provided when forced_photometry=True.")

        pipeline_steps = [
            "background",
            "psf",
            "center forced",
            "isophoteinit forced",
            "isophotefit forced",
            "isophoteextract forced",
            "writeprof",
        ]

        # Add "_fp" to each ID for forced photometry
        ids = [f"{_id}_fp_{forced_profile_filter}" for _id in ids]
    else:
        pipeline_steps = [
            "background",
            "psf",
            "center",
            "isophoteinit",
            "isophotefit",
            "isophoteextract",
            "writeprof",
        ]

    # Adding masking if file provided
    if mask_files is not None:
        pipeline_steps.insert(0, "mask segmentation map")

    # Write AutoProf config
    with open(config_file, "w") as f:
        f.write(dedent(f"""\
import numpy as np
ap_process_mode = "{mode}"
ap_name = {ids}
ap_image_file = {image_files}
"""))
        if mask_files is not None:
            f.write(f"ap_mask_file = {mask_files}\n")

        f.write(dedent(f"""\
ap_saveto = "{out_dir}"
ap_pixscale = {pixelscale}
ap_zeropoint = {zeropoint}
ap_samplegeometricscale = {gscale}
ap_doplot = True
ap_extractfull = True
ap_fluxunits = "{unit_type}"
ap_isoclip = True
ap_isoclip_nsigma = 5
ap_ellipsefit = True
ap_fix_pa = False
ap_initial_pa = 45.0 
ap_fix_ellipticity = False
ap_initial_ellipticity = 0.3
"""))

        if forced_photometry:
            f.write(f'ap_forcing_profile = r"{forced_profile_path}"\n')

        if fourier_orders:
            f.write(f"ap_iso_measurecoefs = {fourier_orders}\n")

        f.write(f"ap_new_pipeline_steps = {pipeline_steps}\n")

    # Run AutoProf
    os.chdir(out_dir)
    try:
        import sys
        if config_name.replace(".py", "") in sys.modules:
            del sys.modules[config_name.replace(".py", "")]

        PIPELINE = Pipeline.Isophote_Pipeline(loggername=log_path)
        PIPELINE.Process_ConfigFile(config_name.replace(".py", ""))

        print(f"AutoProf completed successfully for {ids}!")

    except Exception as e:
        print(f"AutoProf failed for {ids}. Logging error...")

        error_log_file = Path(out_dir) / f"autoprof_error_{ids[0]}.log"
        with open(error_log_file, "w") as log:
            log.write(f"AutoProf failed for {ids}\n")
            log.write(f"Error type: {type(e).__name__}\n")
            log.write(f"Error message: {str(e)}\n")
            log.write("Full traceback:\n")
            log.write(traceback.format_exc())

        print(f"Error log saved to: {error_log_file}")

        

In [12]:
configure_logging(name="nicl.euclid.mask", level="DEBUG")
configure_logging(name="nicl.mask", level="DEBUG")

<Logger nicl.mask (DEBUG)>

In [13]:
# | export

def Extract_SB_using_AP_shapes(
    image_path, object_mask_path=None, profile_path=None, bcg_pos=None, annuli_shape='elliptical', pixelscale=0.3, 
    core_mask_path=None, rad_limit_annulus=None, verbose=None, num_points=1, output_csv_path=None, 
    show_plot=True, plot_output_path=None, prefix=None, plot_lims=1000):
    
    """
    Compute flux statistics for elliptical or circular annuli centred either at bcg_pos (SkyCoord) 
    or at image centre if bcg_pos is not given.
    """

    profile = pd.read_csv(profile_path, skiprows=1)

    ccd = CCDData.read(image_path, unit="adu")
    image = ccd.data
    wcs = ccd.wcs

    image_height, image_width = image.shape

    # If bcg position is not provided image centre is taken as the centre.
    
    if bcg_pos is not None:
        x, y = wcs.world_to_pixel(bcg_pos)
    else:
        x, y = image_width / 2, image_height / 2

    # Loading mask if provided, otherwise use a blank mask
    
    if object_mask_path is not None:
        object_mask = fits.getdata(object_mask_path).astype(bool)
    else:
        object_mask = np.zeros_like(image, dtype=bool)

    if core_mask_path is not None:
        core_mask = fits.getdata(core_mask_path).astype(bool)
        combined_mask = object_mask | core_mask | ~np.isfinite(image)
    else:
        combined_mask = object_mask | ~np.isfinite(image)

    masked_image = np.where(combined_mask, np.nan, image)

    profile['R'] = profile['R'] / pixelscale  # Convert arcsec to pixels

    if rad_limit_annulus:
        profile = profile[profile['R'] < rad_limit_annulus]

    rad = profile['R'].values
    ellip = profile['ellip'].values
    pa = profile['pa'].values

    selected_annuli = sorted(set(rad))
    flux_stats = []
    problematic_annuli = []

    for i in range(len(selected_annuli) - 1):
        r_inner = selected_annuli[i]
        r_outer = selected_annuli[i + 1]
        e = ellip[i]
        theta = np.deg2rad(pa[i])

        if annuli_shape == 'circular':
            annulus = CircularAnnulus((x, y), r_in=r_inner, r_out=r_outer)
            
        else:
            annulus = EllipticalAnnulus(
                (x, y),
                a_in=r_inner,
                a_out=r_outer,
                b_in=r_inner * (1 - e),
                b_out=r_outer * (1 - e),
                theta=theta - np.pi/2  # Convert from PA to photutils angle (the definition of theta is different in AP and photutils/matplotlib)
            )
        annulus_mask = annulus.to_mask(method='center')
        mask_image = annulus_mask.to_image(masked_image.shape)

        valid_flux_values = masked_image[np.isfinite(masked_image) & (mask_image > 0)]

        total_pixels = np.sum(mask_image > 0)
        total_valid = len(valid_flux_values)
        total_masked = total_pixels - total_valid

        if total_valid == 0:
            problematic_annuli.append({'x': x, 'y': y, 'r_inner': r_inner, 'r_outer': r_outer})
            mean_flux = median_flux = std_flux = np.nan
            clipped_mean_flux = clipped_median_flux = clipped_std_flux = np.nan
        else:
            mean_flux = np.nanmean(valid_flux_values) / (pixelscale**2)
            median_flux = np.nanmedian(valid_flux_values) / (pixelscale**2)
            std_flux = np.nanstd(valid_flux_values) if len(valid_flux_values) > 1 else np.nan
            mad_flux = median_abs_deviation(valid_flux_values, scale="normal", nan_policy='omit') if len(valid_flux_values) > 1 else np.nan

            clipped = sigma_clip(valid_flux_values, sigma=3, cenfunc='median', maxiters=5)
            clipped_valid = clipped.data[~clipped.mask]

            clipped_mean_flux = np.nanmean(clipped_valid) / (pixelscale**2)
            clipped_median_flux = np.nanmedian(clipped_valid) / (pixelscale**2)
            clipped_std_flux = np.nanstd(clipped_valid) if len(clipped_valid) > 1 else np.nan
            clipped_mad_flux = median_abs_deviation(clipped_valid, scale="normal", nan_policy='omit') if len(clipped_valid) > 1 else np.nan

        flux_stats.append({
            'Centre_pixel': (x, y),
            'Inner_radius_pix': r_inner,
            'Outer_radius_pix': r_outer,
            'Inner_radius_arcsec': r_inner * pixelscale,
            'Outer_radius_arcsec': r_outer * pixelscale,
            'SMA_annulus_centre_pix': (r_inner + r_outer) / 2,
            'SMA_annulus_centre_arcsec': ((r_inner * pixelscale) + (r_outer* pixelscale)) / 2,
            'Mean_flux_annulus': mean_flux,
            'Median_flux_annulus': median_flux,
            'Std_flux_annulus': std_flux,
            'MAD_flux_annulus': std_flux,
            'Total_valid_pix_annulus': total_valid,
            'Total_masked_pix_annulus': total_masked,
            'Clipped_mean_flux_annulus': clipped_mean_flux,
            'Clipped_median_flux_annulus': clipped_median_flux,
            'Std_clipped_flux_annulus': clipped_std_flux,
            'MAD_clipped_flux_annulus': clipped_std_flux,

        })

    if show_plot:
        fig, ax = plt.subplots(figsize=(6, 6))
        
        ax.imshow(image, origin='lower', cmap='gray', norm=simple_norm(masked_image, 'sqrt', percent=80))

        # Mask overlay
        mask_overlay = np.ma.masked_where(~combined_mask, combined_mask.astype(float))
        ax.imshow(mask_overlay, origin='lower', cmap='Reds', vmin=0, vmax=1, alpha=0.8)
        
        for i in range(len(selected_annuli) - 1):
            r_outer = selected_annuli[i + 1]
            e = ellip[i]
            theta = np.deg2rad(pa[i])

            if annuli_shape == 'circular':
                circle = Circle((x, y), radius=r_outer, edgecolor='blue', facecolor='none', lw=1)
                ax.add_patch(circle)
            else:
                ellipse = Ellipse(
                    (x, y),
                    width=2 * r_outer,
                    height=2 * r_outer * (1 - e),
                    angle=np.rad2deg(theta - np.pi/2),
                    edgecolor='red',
                    facecolor='none',
                    lw=0.5
                )
                ax.add_patch(ellipse)

        ax.set_title("Annuli Overlay")
        if plot_output_path:
            plt.savefig(plot_output_path / f"Annuli_overlay{prefix if prefix else ''}.pdf", dpi=100)
        plt.show()

    flux_stats_df = pd.DataFrame(flux_stats)
    return flux_stats_df, profile, problematic_annuli



def plot_profile(
    prof, 
    x_column='SMA_annulus_centre_pix', #make sure the initial radii are in pixels, otherwise conversions will not be converted correctly
    y_column='Median_flux_annulus',
    x_unit='pix',  # which units the x axis should be converted e,g, 'arcsec' or 'kpc'. default 'pix' no conversion needed.
    pixelscale=0.3,  # arcsec/pixel
    redshift=None,  # Used if x_unit='kpc'
    cosmology=cosmo,  # Astropy cosmology object
    x_label=None,
    y_label=None,
    label = None,
    logy=True,
    logx=True,
    figsize=(6, 5),
    color='tab:blue',
    marker='o',
    markersize=1,
    title=None,
    ax=None,
    save_fig=None,
    plot_dir = None,
    use_y_errors = False,
    Yerr_column = None          
):

    R = prof[x_column]
    Y = prof[y_column]
    if use_y_errors:
        Yerr = prof[Yerr_column]
    if x_unit == 'arcsec':
        R = R * pixelscale
        x_label = x_label or 'Radius (arcsec)'

    elif x_unit == 'kpc':
        if redshift is None:
            raise ValueError("To plot in kpc, please provide a redshift.")
        
        arcsec_to_kpc = cosmo.arcsec_per_kpc_proper(redshift)
        
        R = R * pixelscale / arcsec_to_kpc.value
        x_label = x_label or f'Radius (kpc)'

    else:  # 'pix'
        x_label = x_label or 'Radius (pixels)'

    if ax is None:
        fig, ax = plt.subplots(figsize=figsize)

    if use_y_errors:
        ax.errorbar(R, Y, yerr=Yerr, fmt='o-', markerfacecolor=color, markersize=markersize, label=label)
    
    else:
        ax.plot(R, Y, marker=marker, linestyle='-', color=color, markersize=markersize, label=label)

    ax.set_xlabel(x_label)
    ax.set_ylabel(y_label)

    if logy:
        ax.set_yscale('log')
        ax.set_ylim(bottom=1e-5)
    if logx:
        ax.set_xscale('log')

    if title:
        ax.set_title(title)

    plt.tight_layout()

    if save_fig:
        plt.savefig( plot_dir / "SB_profile_comarison.pdf", dpi=100)
    return ax


In [14]:
def load_profiles_and_measurements(
    outdir,
    cluster_id,
    filter_name,
    prefixes,
    mask_filter_name=None,
    arcsec_to_kpc=None,
    pixelscale=0.3,
    load_measurement_file=False,
    forced_photometry=False,
    forced_profile_filter=None
):

    profiles = []
    measurements = []

    mask_filter_name = mask_filter_name or filter_name

    for prefix in prefixes:
        # Construct label: include mask_filter_name only if different from filter_name
        label_parts = [cluster_id, filter_name]
        
        if mask_filter_name != filter_name:
            label_parts.append(mask_filter_name)
            
        if prefix:
            label_parts.append(prefix)
            
        if forced_photometry:
            if not forced_profile_filter:
                raise ValueError('forced_profile_filter must be defined while forced_photometry=True')
            label_parts.append(f"fp_{forced_profile_filter}")
            
        label = "_".join(label_parts)

        # Construct profile filename similarly
        filename_parts = [cluster_id, filter_name]
        if mask_filter_name != filter_name:
            filename_parts.append(mask_filter_name)
            
        if prefix:
            filename_parts.append(prefix)
            
        if forced_photometry:
            if not forced_profile_filter:
                raise ValueError('forced_profile_filter must be defined while forced_photometry=True')
            filename_parts.append(f"fp_{forced_profile_filter}")
            
        prof_filename = "_".join(filename_parts) + ".prof"
        
        prof_path = outdir / cluster_id / "autoprof_results" / prof_filename

        if prof_path.exists():
            try:
                prof_df = pd.read_csv(prof_path, skiprows=1)
                profiles.append((label, prof_df))
            except Exception as e:
                print(f"[ERROR] Failed to read profile file {prof_path}: {e}")
        else:
            print(f"[INFO] Profile file not found: {prof_path}")

        # Load measurement file (optional)
        if load_measurement_file:
            if forced_photometry:
                if not forced_profile_filter:
                    raise ValueError('forced_profile_filter must be defined while forced_photometry=True')
                label.append(f"_fp_{forced_profile_filter}")
                
            meas_filename = f"Profile_measurements_{label}.csv" if prefix else "Profile_measurements.csv"
            meas_path = outdir / cluster_id / meas_filename

            if meas_path.exists():
                try:
                    meas_df = pd.read_csv(meas_path)
                    measurements.append((label, meas_df))
                except Exception as e:
                    print(f"[ERROR] Failed to read measurement file {meas_path}: {e}")
            else:
                print(f"[INFO] Measurement file not found: {meas_path}")

    if load_measurement_file:
        return profiles, measurements
    else:
        return profiles


In [15]:
def plot_sb_profiles(
    profiles=None,
    measurements=None,
    pixelscale=0.3,
    rad_output_unit='pix',
    cosmology=None,
    redshift=None,
    specific_profile_prefix=None,
    plot_profiles=True,
    plot_measurements=True,
    xlog=True,
    ylog=True,
    ax=None,
    savefig=False,
    outdir=None,
    prefix_colors=None,
    legendloc='lower left',
    legendfontsize=10

):
    if ax is None:
        fig, ax = plt.subplots(figsize=(7, 5))

    if prefix_colors is None:
        prefix_colors = {}

    allowed_labels = set(specific_profile_prefix) if specific_profile_prefix else None

    # Unit conversion
    if rad_output_unit == 'pix':
        scale_conversion = 1 / pixelscale
    elif rad_output_unit == 'arcsec':
        scale_conversion = 1
    elif rad_output_unit == 'kpc':
        if cosmology is None or redshift is None:
            raise ValueError("Cosmology and redshift must be provided for kpc conversion.")
        scale_conversion = 1 / cosmology.arcsec_per_kpc_proper(redshift).value
    else:
        raise ValueError(f"Unsupported rad_output_unit: {rad_output_unit}")

    # Default styles
    styles = {
        'isophote': {'linestyle': '-', 'marker': 'o'},
        'annuli': {'linestyle': '--', 'marker': 's'},
        'external': {'linestyle': ':', 'marker': '^'}
    }

    # Internal tracking of colors
    label_colors = {}
    color_cycle = plt.rcParams['axes.prop_cycle'].by_key()['color']
    color_index = 0

    def extract_prefix(label):
        """Extract prefix from label assuming format: cluster_filter_maskfilter_prefix"""
        parts = label.split("_")
        return "_".join(parts[3:]) if len(parts) > 3 else "_".join(parts[2:])

    def get_color(label):
        prefix = extract_prefix(label)
        if prefix in prefix_colors:
            return prefix_colors[prefix]
        nonlocal color_index
        if label not in label_colors:
            label_colors[label] = color_cycle[color_index % len(color_cycle)]
            color_index += 1
        return label_colors[label]

    def get_marker_style(base_marker):
        return "x" if "fp" in label else base_marker

    # Plot .prof profiles
    if plot_profiles and profiles:
        for item in profiles:
            if isinstance(item, tuple) and len(item) == 2:
                label, prof = item
            else:
                raise ValueError("Each item in 'profiles' must be a (label, DataFrame) tuple")

            if allowed_labels and label not in allowed_labels:
                continue

            color = get_color(label)

            # isophote
            if 'R' in prof and 'I' in prof:
                ax.plot(
                    prof['R'] * scale_conversion,
                    prof['I'],
                    label=f"AP Isophote - {label}",
                    color=color,
                    linestyle=styles['isophote']['linestyle'],
                    marker=get_marker_style(styles['isophote']['marker']),
                    ms=3,
                    alpha=0.7
                )

            # annuli
            if 'SMA_annulus_centre' in prof and 'Median_flux_annulus' in prof:
                ax.plot(
                    prof['SMA_annulus_centre'] * scale_conversion,
                    prof['Median_flux_annulus'],
                    label=f"AP Annuli - {label}",
                    color=color,
                    linestyle=styles['annuli']['linestyle'],
                    marker=get_marker_style(styles['annuli']['marker']),
                    ms=5,
                    alpha=0.7
                )

    # Plot measurement files
    if plot_measurements and measurements:
        for item in measurements:
            if isinstance(item, tuple) and len(item) == 2:
                label, meas = item
            else:
                raise ValueError("Each item in 'measurements' must be a (label, DataFrame) tuple")

            if allowed_labels and label not in allowed_labels:
                continue

            color = get_color(label)

            if 'SMA_annulus_centre_arcsec' not in meas:
                raise KeyError("Expected 'SMA_annulus_centre_arcsec' in measurement data.")

            radius = meas['SMA_annulus_centre_arcsec'] * scale_conversion

            ax.plot(
                radius,
                meas["Clipped_median_flux_annulus"],
                label=f"External Annuli - {label}",
                color=color,
                linestyle=styles['external']['linestyle'],
                marker=get_marker_style(styles['external']['marker']),
                ms=10,
                alpha=0.7
            )

    ax.set_xlabel(f"R ({rad_output_unit})")
    ax.set_ylabel(r"Surface Brightness (flux / arcsec$^2$)")
    if xlog:
        ax.set_xscale('log')
    if ylog:
        ax.set_yscale('log')
    ax.legend(loc=legendloc, fontsize=legendfontsize)

    if savefig:
        if outdir is None:
            raise ValueError("You must specify `outdir` when `savefig=True`.")
        Path(outdir).mkdir(exist_ok=True)
        label = "SB_profile_comparisons.pdf"
        plt.savefig(Path(outdir) / label)

    return ax

In [16]:
def plot_residual_comparison(
    reference_label,
    reference_from="profiles",   # "profiles" or "measurements"
    compare_from="profiles",     # "profiles" or "measurements"
    method="isophote",           # "isophote" or "annulus"
    profiles=None,
    measurements=None,
    prefix_colors=None,
    ax=None,
    xlims=[None, None],
    ylims=[None, None],
    savefig=False,
    outdir = None,
    legendloc='lower left',
    legendfontsize=10
):
    if prefix_colors is None:
        prefix_colors = {}

    # Defining which radius and flux columns to use
    def get_columns(source_type, method):
        if source_type == "profiles":
            if method == "isophote":
                return "R", "I"
            elif method == "annulus":
                return "SMA_annulus_centre", "Median_flux_annulus"
            else:
                raise ValueError("Invalid method. Use 'isophote' or 'annulus'")
        elif source_type == "measurements":
            return "SMA_annulus_centre_arcsec", "Clipped_median_flux_annulus"
        else:
            raise ValueError("source_type must be 'profiles' or 'measurements'")

    ref_r_col, ref_flux_col = get_columns(reference_from, method)
    cmp_r_col, cmp_flux_col = get_columns(compare_from, method)

    # whether using the .prof or measurement file
    if reference_from == "profiles":
        if profiles is None:
            raise ValueError("You must provide `profiles` when using 'profiles' as reference.")
        ref_data = dict(profiles)
    else:
        if measurements is None:
            raise ValueError("You must provide `measurements` when using 'measurements' as reference.")
        ref_data = dict(measurements)

    if reference_label not in ref_data:
        raise ValueError(f"Reference label '{reference_label}' not found in {reference_from}!")

    ref_df = ref_data[reference_label]
    ref_r = ref_df[ref_r_col].values
    ref_flux = ref_df[ref_flux_col].values

    # Load comparison data
    if compare_from == "profiles":
        if profiles is None:
            raise ValueError("You must provide `profiles` when using 'profiles' as comparison data.")
        compare_data = profiles
    else:
        if measurements is None:
            raise ValueError("You must provide `measurements` when using 'measurements' as comparison data.")
        compare_data = measurements


    if ax is None:
        fig, ax = plt.subplots(figsize=(7, 5))

    for label, df in compare_data:
        if label == reference_label and compare_from == reference_from:
            continue

        if cmp_r_col not in df.columns or cmp_flux_col not in df.columns:
            print(f"Skipping {label} due to missing columns.")
            continue

        other_r = df[cmp_r_col].values
        other_flux = df[cmp_flux_col].values

        ref_interp_flux = np.interp(other_r, ref_r, ref_flux)
        residual_fraction = (ref_interp_flux - other_flux) / ref_interp_flux

        ax.plot(
            other_r,
            residual_fraction,
            label=f"{label} vs {reference_label}",
            linestyle="-",
            marker="o",
            ms=3,
            alpha=0.7
        )

    ax.axhline(0, color="gray", linestyle="--", lw=1)
    ax.set_xscale("log")
    ax.set_xlabel("Radius")
    ax.set_ylabel(r'$\left(SB_{\rm ref} - SB_{\rm comp}\right) / SB_{\rm ref}$')
    ax.set_title(f"Fractional Residuals in SB (method: {method} result comparisons)")
    ax.legend(loc=legendloc, fontsize=legendfontsize)

    if xlims:
        ax.set_xlim(xlims[0], xlims[1])
    if ylims:
        ax.set_ylim(ylims[0], ylims[1])
    else:
        ax.set_ylim(-0.1, 0.1)

    plt.tight_layout()

    if savefig:
        from pathlib import Path
        label = f"SB_fractional_change_{compare_from}_vs_{reference_from}_{method}.pdf"
        Path(outdir).mkdir(exist_ok=True)
        plt.savefig(f"{outdir}/{label}")

    plt.show()


# Below, the main function to wrap above functions that can process real or mock cluster images. 
> It performs optional masking, alternatively can load a existing mask.
> 
> It then performs NaN cleaning for autoprof preperation, then runs autoprof on cleaned image.
> 
> Then using autoprof profiles, it performs SB extraction by measuring median flux within annular rings. It constructs plots, if asked. The constructed SB profile measurements are added to the .prof file.

In [ ]:
# | export

def process_cluster_pipeline(
    image_dir,
    outdir,
    cluster_ids,
    filters,
    mask_filter=None,
    cluster_info_table=None,
    mock_image=False,
    masking=True,
    rerun_masking=True,
    prefix=None,
    prefix2=None,
    run_autoprof_function=True,
    SB_extraction=False,
    show_profiles=False,
    forced_photometry=False,
    forced_profile_filter = None,
    forced_profile_path=None  
):
    """
    A combined function for real and mock clusters with optional masking and mask reuse. 
    Further cleaning to prep images for autoprof, and running autoprof on cleaned images.
    Alternatively, autoprof can also be turned off to only perform SB extraction.
    """
    image_dir = Path(image_dir)
    outdir = Path(outdir)

    for cluster_id in cluster_ids:
        cluster_output_dir = outdir / cluster_id
        cluster_output_dir.mkdir(parents=True, exist_ok=True)

        if not mock_image:
            cluster_info = cluster_info_table[cluster_info_table.NAME == cluster_id].reset_index(drop=True)
            cluster_z = cluster_info.BEST_Z[0]
            bcg_pos = SkyCoord(cluster_info.RA_BCG[0] * u.deg, cluster_info.DEC_BCG[0] * u.deg, frame="icrs")
        else:
            cluster_z = 0.3
            bcg_pos = None

        for filter in filters:
            actual_mask_filter = mask_filter if mask_filter is not None else filter
            print(f"Using {filter} band image and {actual_mask_filter} band mask for {cluster_id}")

            image_label = f"{cluster_id}_{filter}"
            mask_label = f"{cluster_id}_{actual_mask_filter}"

            combined_label = ""
            if prefix:
                combined_label += f"_{prefix}"
            if prefix2:
                combined_label += f"_{prefix2}"

            combined_image_label = f"{image_label}_{actual_mask_filter}" if mask_filter else image_label
            if prefix:
                combined_image_label += f"_{prefix}"
            if prefix2:
                combined_image_label += f"_{prefix2}"

            combined_mask_label = f"{mask_label}{combined_label}"

            if prefix:
                image_filename = f"{image_label}_{prefix}.fits" if mock_image else f"EUC_NIR_W-STK_{filter}-{cluster_id}.fits"
            else:
                image_filename = f"{image_label}.fits" if mock_image else f"EUC_NIR_W-STK_{filter}-{cluster_id}.fits"

            fn = image_dir / image_filename
            print(f"Processing {fn}")

            image = CCDData.read(fn, unit="adu")

            if masking and rerun_masking:
                print("Creating new masks...")

                masks = create_masks(
                    image,
                    z=cluster_z,
                    filter=actual_mask_filter,
                    centre_pos=bcg_pos,
                    make_faint_mask=True,
                    zeropoint="ZPAB",
                )

                masks["background"] = masks["badpixel"] | masks["icl"] | masks["object"]
                masks["measurement"] = masks["badpixel"] | masks["object"] | masks["faint"]

                wcs_header = image.wcs.to_header()
                combined_header = image.meta.copy()
                combined_header.update(wcs_header)

                selected_masks = {k: masks[k] for k in ["background", "measurement", 'icl', 'bcg']}
                save_masks(selected_masks, cluster_output_dir, reference_header=combined_header, label=combined_mask_label)

                print("Masking completed.")
                
            elif masking:
                print("Using existing masks — skipping mask generation.")
            else:
                print("No masking will be applied at all.")

            mask_file = cluster_output_dir / f"{combined_mask_label}_measurement_mask.fits"

            
            AP_results_dir = cluster_output_dir / "autoprof_results"
            AP_results_dir.mkdir(parents=True, exist_ok=True)
            
            if run_autoprof_function:
                if forced_photometry:
                    print(f'Autoprof will be run with fixed photometry, profile {forced_profile_path}')
                
                print("Cleaning image and writing cleaned version...")
                with fits.open(fn) as hdul:
                    hdu_index = 1
                    image_data = hdul[hdu_index].data
                    image_header = hdul[hdu_index].header
    
                image_data_cleaned = np.where(~np.isfinite(image_data), 99, image_data)
                cleaned_name = f"cleaned_{combined_image_label}.fits"
                fits.writeto(AP_results_dir / cleaned_name, image_data_cleaned, header=image_header, overwrite=True)

            
                print(f"Autoprof running on {cleaned_name}, with mask {combined_mask_label}_measurement_mask.fits ...")

            
                run_autoprof(
                    ids=[combined_image_label],
                    image_files=[str(AP_results_dir / cleaned_name)],
                    mask_files=[str(mask_file)] if masking else None,
                    out_dir=str(AP_results_dir),
                    fourier_orders=(1, 4),
                    forced_photometry=forced_photometry, 
                    forced_profile_filter = forced_profile_filter,
                    forced_profile_path=str(forced_profile_path) if forced_photometry else None
                )

                ###### Cleaning some of the excess diagnostic plots ....

                always_keep = {"basic_config.py"}
                jpg_keywords_to_keep = [
                    "Background_hist_",
                    "mask_",
                    "initialize_ellipse_",
                    "photometry_",
                    "photometry_ellipse_",
                ]
                valid_suffixes_to_keep = [".prof", ".aux"]

                for f in AP_results_dir.iterdir():
                    if f.is_file():
                        name = f.name
                        if name in always_keep:
                            continue
                        if any(name.endswith(suffix) for suffix in valid_suffixes_to_keep):
                            continue
                        if name.endswith(".jpg") and any(kw in name for kw in jpg_keywords_to_keep):
                            continue
                        try:
                            f.unlink()
                            print(f"Deleted: {name}")
                        except Exception as e:
                            print(f"Could not delete {name}: {e}")

            if SB_extraction:
                print(f"Extracting SB profile for {cluster_id} in {filter}...")
                if forced_photometry:
                    if not forced_profile_filter:
                        raise ValueError('forced_profile_filter must be provided if forced_photometry=True')
                    combined_image_label += f"_fp_{forced_profile_filter}"
                
                prof_filename = f"{combined_image_label}.prof"
                prof_path = AP_results_dir / prof_filename
                print(f"Loaded Autoprof profile {prof_path}")

                if masking:
                    mask_path = cluster_output_dir / f"{combined_mask_label}_measurement_mask.fits"
                    if not mask_path.exists():
                        print(f"Warning: Mask file not found at {mask_path}, proceeding without mask.")
                        mask_path = None
                else:
                    mask_path = None

                image_path = fn
                plot_output_path = cluster_output_dir

                flux_measurements, profile, problematic_annuli = Extract_SB_using_AP_shapes(
                    image_path=image_path,
                    object_mask_path=str(mask_path) if mask_path else None,
                    profile_path=prof_path,
                    bcg_pos=bcg_pos,
                    plot_output_path=plot_output_path,
                    prefix=combined_image_label
                )

                flux_outfile = cluster_output_dir / f"Profile_measurements_{combined_image_label}.csv"
                flux_measurements.to_csv(flux_outfile, index=False)

                # Appending flux measurements from annuli method to .prof file
                with open(prof_path, 'r') as f:
                    units_line = f.readline().strip()
                    column_line = f.readline().strip()
                    data_df = pd.read_csv(f, skiprows=0) 
                
                unit_list = units_line.lstrip("#").split(",")
                column_list = column_line.split(",")
                
                data_df['SMA_annulus_centre'] = flux_measurements['SMA_annulus_centre_arcsec']
                data_df['Median_flux_annulus'] = flux_measurements['Clipped_median_flux_annulus']
                data_df['MAD_median_flux_annulus'] = flux_measurements['MAD_clipped_flux_annulus']
                
                column_list += ['SMA_annulus_centre', 'Median_flux_annulus', 'MAD_median_flux_annulus']
                unit_list += ['arcsec', 'flux*arcsec^-2', 'flux*arcsec^-2']
                
                with open(prof_path, 'w') as f:
                    f.write("#" + ",".join(unit_list) + "\n")
                    f.write(",".join(column_list) + "\n")
                    data_df.to_csv(f, index=False, header=False)

                print('Annuli measurements are added to the .prof file')
                
                if show_profiles:
                    print('Proceeding for diagnostic plots...')
                    fig, ax = plt.subplots(1, 1)
                    plot_profile(flux_measurements, x_column='SMA_annulus_centre_pix', y_column='Clipped_median_flux_annulus',
                        x_unit='kpc', redshift=cluster_z, y_label='Median Flux', label='Median Flux Annuli', color='tab:green', ax=ax)

                    plot_profile(profile, x_column='R', y_column='I', x_unit='kpc', redshift=cluster_z,
                        y_label='Median Flux', label='Median Flux Isophote (AP)', ax=ax)

                    fig_path = cluster_output_dir / f"SB_comparisons_{combined_image_label}.pdf"
                    plt.savefig(fig_path, dpi=100)
                    plt.show()

                    fig, ax = plt.subplots(1, 2, figsize=(9, 4))
                    plot_profile(profile, x_column='R', y_column='ellip', Yerr_column='ellip_e',
                        x_unit='kpc', redshift=cluster_z, y_label='Ellipticity', ax=ax[0], use_y_errors=True, logy=False, label=combined_image_label)
                    ax[0].set_ylim(0, 1)

                    plot_profile(profile, x_column='R', y_column='pa', Yerr_column='pa_e',
                        x_unit='kpc', redshift=cluster_z, y_label='Position Angle', ax=ax[1], use_y_errors=True, logy=False, label=combined_image_label)
                    ax[1].set_ylim(0, 180)

                    fig_path = cluster_output_dir / f"Shapes_{combined_image_label}.pdf"
                    plt.savefig(fig_path, dpi=100)


# Example runs for a mock cluster
# 1- mock cluster image but no masking is neded, autoprof is running. 

In [ ]:
cluster_ids = ["cluster"]
image_dir = Path("/home/ppzjbg/JGM_Tests/Mock_Images/")
outdir = Path("/home/ppztk1/Erosita/Outputs_Clusters/")
filters = ["VIS"]

process_cluster_pipeline(
    image_dir=image_dir,
    outdir=outdir,
    cluster_ids=cluster_ids,
    filters=filters,
    mock_image=True,
    masking=False,
    rerun_masking = False,
    run_autoprof_function = True, # if .prof files already exist and only SB extraction is needed.
    SB_extraction=True,
    show_profiles = True,
    prefix = 'no_noise' # to run this on 'no_noise' mock images comment this out.
)


## 2- same mock cluster but using a mask loaded from a random cluster. So no rerunning the masking procedure.


In [ ]:

cluster_ids = ["cluster"]
image_dir = Path("/home/ppzjbg/JGM_Tests/Mock_Images/")
outdir = Path("/home/ppztk1/Erosita/Outputs_Clusters/")
filters = ["H"]

process_cluster_pipeline(
    image_dir=image_dir,
    outdir=outdir,
    cluster_ids=cluster_ids,
    filters=filters,
    mock_image=True,
    masking=True,
    rerun_masking = False,
    run_autoprof_function = True, # if .prof files already exist and only SB extraction is needed... set False
    SB_extraction=True,
    show_profiles = True,
    prefix2='loaded_mask'
)

# 3- same mock cluster in a skypatch with sources, so we will perform masking.


In [ ]:
cluster_ids = ["cluster"]
image_dir = Path("/home/ppzjbg/JGM_Tests/Mock_Images/")
outdir = Path("/home/ppztk1/Erosita/Outputs_Clusters/")
filters = ["VIS"]

process_cluster_pipeline(
    image_dir=image_dir,
    outdir=outdir,
    cluster_ids=cluster_ids,
    filters=filters,
    mock_image=True,
    masking=True,
    rerun_masking=True,
    run_autoprof_function = True,
    SB_extraction=True,
    show_profiles = True,
    prefix='random_field',
    prefix2=f'growth_{FAINT_GROWTH}'
)

# 4- Below, runs the pipeline for a J band image, uses a mask and forced photometry profile from an H band image.

In [ ]:


# 4- Below, runs the pipeline for a J band image, uses a mask and forced photometry profile from an H band image.


cluster_ids = ["cluster"]
image_dir = Path("/home/ppzjbg/JGM_Tests/Mock_Images/")
outdir = Path("/home/ppztk1/Erosita/Outputs_Clusters/")
forced_profile_file = Path("/home/ppztk1/Erosita/Outputs_Clusters/cluster/autoprof_results/cluster_H_random_field_growth_0.25.prof")

filters = ["J"]

process_cluster_pipeline(
    image_dir=image_dir,
    outdir=outdir,
    cluster_ids=cluster_ids,
    filters=filters,
    mask_filter='H',
    mock_image=True,
    masking=True,
    rerun_masking=False,
    run_autoprof_function = True, # if .prof files already exist and only SB extraction is needed.
    SB_extraction=True,
    show_profiles = True,
    prefix='random_field',
    prefix2=f'growth_{FAINT_GROWTH}',
    forced_photometry=True,
    forced_profile_filter='H',
    forced_profile_path=forced_profile_file
)

# Plotting multiple profiels and comparisons

In [ ]:
outdir = Path(f'/home/ppztk1/Erosita/Outputs_Clusters/')
cluster_id = 'cluster'
plotdir = outdir / cluster_id

pixelscale = 0.3 
z = 0.1
arcsec_to_kpc = cosmo.arcsec_per_kpc_proper(z).value

prefixes = ["no_noise", "random_field_growth_0.25", "random_field_growth_0.5", "random_field_growth_1"]

# H-band image results with H band mask
h_profiles = load_profiles_and_measurements(
    outdir=Path(outdir),
    cluster_id = cluster_id,
    filter_name='H',
    prefixes=prefixes,
)

# J-band image results with H band mask and forced photometry applied from H band dedcued profiles
prefixes = ["random_field_growth_0.25"]

j_profiles = load_profiles_and_measurements(
    outdir=Path(outdir),
    cluster_id=cluster_id,
    filter_name='J',
    mask_filter_name='H',
    prefixes=prefixes,
    forced_photometry=True,
    forced_profile_filter='H'
)

# Vis image results.
prefixes = ["no_noise", "random_field_growth_0.25"]

vis_profiles = load_profiles_and_measurements(
    outdir=Path(outdir),
    cluster_id=cluster_id,
    filter_name='VIS',
    prefixes=prefixes,
)


j_dict = dict(j_profiles)
h_dict = dict(h_profiles)
v_dict = dict(vis_profiles)

print(h_dict.keys(), j_dict.keys(),v_dict.keys())


In [ ]:
fig, ax = plt.subplots(1,1,figsize=(7,5))

plot_sb_profiles(profiles=h_profiles, 
                 ax=ax, 
                 cosmology=cosmo, 
                 redshift=z, 
                 rad_output_unit='kpc', 
                 savefig=True, 
                 outdir=plotdir)
plt.show()

fig, ax = plt.subplots(1,1,figsize=(7,5))
plot_sb_profiles(profiles=j_profiles, 
                 ax=ax,
                 cosmology=cosmo, 
                 redshift=0.1, 
                 rad_output_unit='kpc', 
                 savefig=False, 
                 outdir=plotdir)
plt.show()


fig, ax = plt.subplots(1,1,figsize=(7,5))
plot_sb_profiles(profiles=vis_profiles, 
                 ax=ax, 
                 cosmology=cosmo, 
                 redshift=z, 
                 rad_output_unit='kpc', 
                 savefig=True, 
                 outdir=plotdir)
plt.show()

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(5,4))


plot_sb_profiles(profiles=h_profiles, 
                 ax=ax, 
                 specific_profile_prefix=['cluster_H_random_field_growth_0.25'], 
                 savefig=False, 
                 outdir=plotdir,
                 legendloc='upper right',
                 legendfontsize=8)

plot_sb_profiles(profiles=j_profiles, 
                 ax=ax, 
                 savefig=False, 
                 outdir=plotdir,
                 legendloc='upper right',
                 legendfontsize=8)


plt.show()

In [ ]:
prefix_colors = {
    "fluctuations": "tab:orange",
    "no_noise": "teal",
    "loaded_mask": "tab:green",
    "random_field": "hotpink",
    "random_field_growth_0.25": "red",
    "random_field_growth_0.5": "gold",
    "random_field_growth_1": "salmon",
    "random_field_growth_1.25": "purple"
}

fig, ax = plt.subplots(1,1,figsize=(7,5))
plot_sb_profiles(
    profiles=h_profiles,
    cosmology=cosmo,
    redshift=0.1,
    rad_output_unit='kpc',
    specific_profile_prefix=[
        "cluster_H_random_field_growth_0.25",
        "cluster_H_random_field_growth_0.5"
    ],
    prefix_colors=prefix_colors,
    savefig=True,
    outdir=plotdir, 
    ax=ax
)

plot_sb_profiles(
    profiles=j_profiles,
    cosmology=cosmo,
    redshift=0.1,
    rad_output_unit='kpc',
    specific_profile_prefix=[
        "cluster_J_H_random_field_growth_0.25_fp_H",
    ],
    prefix_colors=prefix_colors,
    savefig=True,
    outdir=plotdir, 
    ax=ax,
    legendloc='lower left'
)

# The shape profiles of different runs

In [ ]:
profiles= h_profiles
fig, ax = plt.subplots(1, 2, figsize=(7, 4), sharex=True)


for label, prof in profiles:
    ax[0].errorbar(
        prof["R"], prof["ellip"], yerr=prof["ellip_e"],
        label=label,
        fmt='o-',
        ms=3, alpha=0.7
    )
    ax[1].errorbar(
        prof["R"], prof["pa"], yerr=prof["pa_e"],
        label=label,
        fmt='o-',
        ms=3, alpha=0.7
    )

ax[0].set_xlabel("R")
ax[0].set_ylabel("Ellipticity")
ax[0].set_ylim(0, 1)
ax[0].legend()
ax[0].set_xscale('log')


ax[1].set_xlabel("R [kpc]")
ax[1].set_ylabel("Position Angle [deg]")
ax[1].set_ylim(0, 180)
ax[1].legend()
ax[1].set_xscale('log')

fig.suptitle("Shape Profiles")
plt.tight_layout()
plt.show()


In [50]:
from pathlib import Path
from astropy.nddata import CCDData
from astropy.coordinates import SkyCoord
from astropy import units as u
import numpy as np
import pandas as pd
from astropy.io import fits
import matplotlib.pyplot as plt

def process_cluster_pipeline_updated(
    image_dir,
    outdir,
    cluster_ids,
    filters,
    mask_filter=None,
    cluster_info_table=None,
    mock_image=False,
    masking=True,
    rerun_masking=True,
    prefix=None,
    prefix2=None,
    run_autoprof_function=True,
    SB_extraction=False,
    show_profiles=False,
    forced_photometry=False,
    forced_profile_filter=None,
    forced_profile_path=None,
    external_mask_path=None
):
    image_dir = Path(image_dir)
    outdir = Path(outdir)

    for cluster_id in cluster_ids:
        cluster_output_dir = outdir / cluster_id
        cluster_output_dir.mkdir(parents=True, exist_ok=True)

        if not mock_image:
            cluster_info = cluster_info_table[cluster_info_table.NAME == cluster_id].reset_index(drop=True)
            cluster_z = cluster_info.BEST_Z[0]
            bcg_pos = SkyCoord(cluster_info.RA_BCG[0] * u.deg, cluster_info.DEC_BCG[0] * u.deg, frame="icrs")
        else:
            cluster_z = 0.1
            bcg_pos = None

        for filter in filters:
            actual_mask_filter = mask_filter if mask_filter is not None else filter
            print(f"Using {filter} band image and {actual_mask_filter} band mask for {cluster_id}")

            image_label = f"{cluster_id}_{filter}"
            combined_label = ""
            if prefix:
                combined_label += f"_{prefix}"
            if prefix2:
                combined_label += f"_{prefix2}"

            combined_image_label = f"{image_label}_{actual_mask_filter}" if mask_filter else image_label
            combined_image_label += combined_label
            combined_mask_label = f"{cluster_id}_{actual_mask_filter}{combined_label}"


            if filter in ["H","J","Y"]:
                image_filename = f"{image_label}_{prefix}.fits" if mock_image else f"EUC_NIR_W-STK_{filter}-{cluster_id}.fits"
            
            elif filter in ["YJH"]:
                image_filename = f"{image_label}_{prefix}.fits" if mock_image else f"{cluster_id}_NIR_YJH_COADDED.fits"
            
            elif filter in ["VIS"]:
                image_filename = f"{image_label}_{prefix}.fits" if mock_image else f"EUC_VIS_SWL-STK-{cluster_id}.fits"

            fn = image_dir / image_filename
            print(f"Processing {fn}")

            if masking and rerun_masking:
                
                if actual_mask_filter in ["H", "J", "Y", "YJH"]:
                                    
                    label = f"{cluster_id}_{prefix}" if prefix else f"{cluster_id}"
                    nir_mask_path = cluster_output_dir / f"{label}_NIR_measurement_mask.fits"
                    default_mask_path = nir_mask_path
                    
                    if not nir_mask_path.exists():
                        print("Creating new NIR masks...")
                        if mock_image:
                            image_H = image_dir / f"{cluster_id}_H_{prefix}.fits"
                            image_J = image_dir / f"{cluster_id}_J_{prefix}.fits"
                            image_Y = image_dir / f"{cluster_id}_Y_{prefix}.fits"
                            
                        else:
                            image_H = image_dir / f"EUC_NIR_W-STK_H-{cluster_id}.fits"
                            image_J = image_dir / f"EUC_NIR_W-STK_J-{cluster_id}.fits"
                            image_Y = image_dir / f"EUC_NIR_W-STK_Y-{cluster_id}.fits"

                        create_combined_nir_mask(
                            str(image_H), str(image_J), str(image_Y),
                            centre_pos=bcg_pos,
                            redshift=cluster_z,
                            filter=filter,
                            label=label,
                            output_dir=str(cluster_output_dir)
                        )
                        
                        print("Created NIR masks!")
                        
                    else:
                        print("NIR masks already exist. Skipping creation.")

                elif actual_mask_filter == "VIS":

                    label = f"{cluster_id}_{prefix}" if prefix else f"{cluster_id}"
                    vis_mask_path = cluster_output_dir / f"{label}_VIS_measurement_mask.fits"
                    default_mask_path = vis_mask_path
                    
                    
                    if not vis_mask_path.exists():
                        print("Creating new VIS masks...")

                        create_vis_mask(
                            str(fn),
                            centre_pos=bcg_pos,
                            redshift=cluster_z,
                            filter=filter,
                            label=label,
                            output_dir=str(cluster_output_dir)
                        )
            
                        print("Created VIS mask!")
                    else:
                        print("VIS masks already exist. Skipping creation.")
                else:
                    raise ValueError(f"Unknown mask filter: {actual_mask_filter}")
                    

            AP_results_dir = cluster_output_dir / "autoprof_results"
            AP_results_dir.mkdir(parents=True, exist_ok=True)

            if masking:
                mask_path = Path(external_mask_path) if external_mask_path is not None else default_mask_path

                if not mask_path.exists():
                    raise FileNotFoundError(f"Expected mask not found at {mask_path}")

                temp_cleaned_dir = cluster_output_dir / "temp_cleaned"
                temp_cleaned_dir.mkdir(exist_ok=True)

                cleaned_paths = create_bkgsub_images(
                    image_paths={filter: str(fn)},
                    measurement_mask_path=str(mask_path),
                    output_background_dir=cluster_output_dir,
                    temp_cleaned_dir=temp_cleaned_dir,
                    label=combined_image_label,
                    clean_nans=True
                )
                
                cleaned_image_path = cleaned_paths[filter]
            else:
                with fits.open(fn) as hdul:
                    hdu_index = 1 if len(hdul) > 1 else 0
                    image_data = hdul[hdu_index].data
                    image_header = hdul[hdu_index].header
                image_data_cleaned = np.where(~np.isfinite(image_data), 99, image_data)
                cleaned_image_path = cluster_output_dir / f"cleaned_{combined_image_label}.fits"
                fits.writeto(cleaned_image_path, image_data_cleaned, header=image_header, overwrite=True)

            if run_autoprof_function:
                if forced_photometry:
                    print(f"Autoprof will be run with fixed photometry, profile {forced_profile_path}")

                print(f"Autoprof running on {cleaned_image_path}, with mask {mask_path.name if masking else 'None'} ...")
                run_autoprof(
                    ids=[combined_image_label],
                    image_files=[str(cleaned_image_path)],
                    mask_files=[str(mask_path)] if masking else None,
                    out_dir=str(AP_results_dir),
                    fourier_orders=(1, 4),
                    forced_photometry=forced_photometry,
                    forced_profile_filter=forced_profile_filter,
                    forced_profile_path=str(forced_profile_path) if forced_photometry else None
                )
                                
                
                ###### Cleaning some of the excess diagnostic plots ....

                always_keep = {"basic_config.py"}
                jpg_keywords_to_keep = [
                    "Background_hist_",
                    "mask_",
                    "initialize_ellipse_",
                    "photometry_",
                    "photometry_ellipse_",
                ]
                valid_suffixes_to_keep = [".prof", ".aux"]

                for f in AP_results_dir.iterdir():
                    if f.is_file():
                        name = f.name
                        if name in always_keep:
                            continue
                        if any(name.endswith(suffix) for suffix in valid_suffixes_to_keep):
                            continue
                        if name.endswith(".jpg") and any(kw in name for kw in jpg_keywords_to_keep):
                            continue
                        try:
                            f.unlink()
                            print(f"Deleted: {name}")
                        except Exception as e:
                            print(f"Could not delete {name}: {e}")

            if SB_extraction:
                print(f"Extracting SB profile for {cluster_id} in {filter}...")
                if forced_photometry and not forced_profile_filter:
                    raise ValueError('forced_profile_filter must be provided if forced_photometry=True')

                prof_filename = f"{combined_image_label}.prof"
                prof_path = AP_results_dir / prof_filename
                
                image_path = fn

                flux_measurements, profile, problematic_annuli = Extract_SB_using_AP_shapes(
                    image_path=image_path,
                    object_mask_path=str(mask_path) if masking else None,
                    profile_path=prof_path,
                    bcg_pos=bcg_pos,
                    plot_output_path=cluster_output_dir,
                    prefix=combined_image_label
                )

                flux_outfile = cluster_output_dir / f"Profile_measurements_{combined_image_label}.csv"
                flux_measurements.to_csv(flux_outfile, index=False)

                with open(prof_path, 'r') as f:
                    units_line = f.readline().strip()
                    column_line = f.readline().strip()
                    data_df = pd.read_csv(f, skiprows=0)

                unit_list = units_line.lstrip("#").split(",")
                column_list = column_line.split(",")

                data_df['SMA_annulus_centre'] = flux_measurements['SMA_annulus_centre_arcsec']
                data_df['Median_flux_annulus'] = flux_measurements['Clipped_median_flux_annulus']
                data_df['MAD_median_flux_annulus'] = flux_measurements['MAD_clipped_flux_annulus']

                column_list += ['SMA_annulus_centre', 'Median_flux_annulus', 'MAD_median_flux_annulus']
                unit_list += ['arcsec', 'flux*arcsec^-2', 'flux*arcsec^-2']

                with open(prof_path, 'w') as f:
                    f.write("#" + ",".join(unit_list) + "\n")
                    f.write(",".join(column_list) + "\n")
                    data_df.to_csv(f, index=False, header=False)

                print('Annuli measurements are added to the .prof file')

                if show_profiles:
                    fig, ax = plt.subplots(1, 1)
                    plot_profile(flux_measurements, x_column='SMA_annulus_centre_pix', y_column='Clipped_median_flux_annulus',
                                 x_unit='kpc', redshift=cluster_z, y_label='Median Flux', label='Median Flux Annuli', ax=ax)

                    plot_profile(profile, x_column='R', y_column='I', x_unit='kpc', redshift=cluster_z,
                                 y_label='Median Flux', label='Median Flux Isophote (AP)', color='tab:green', ax=ax)

                    fig_path = cluster_output_dir / f"SB_comparisons_{combined_image_label}.pdf"
                    plt.savefig(fig_path, dpi=100)
                    plt.show()

                    fig, ax = plt.subplots(1, 2, figsize=(9, 4))
                    plot_profile(profile, x_column='R', y_column='ellip', Yerr_column='ellip_e',
                                 x_unit='kpc', redshift=cluster_z, y_label='Ellipticity', ax=ax[0], use_y_errors=True, logy=False, color='tab:green', label=combined_image_label)
                    ax[0].set_ylim(0, 1)

                    plot_profile(profile, x_column='R', y_column='pa', Yerr_column='pa_e',
                                 x_unit='kpc', redshift=cluster_z, y_label='Position Angle', ax=ax[1], use_y_errors=True, logy=False, color='tab:green', label=combined_image_label)
                    ax[1].set_ylim(0, 180)

                    fig_path = cluster_output_dir / f"Shapes_{combined_image_label}.pdf"
                    plt.savefig(fig_path, dpi=100)
                    plt.show()


In [ ]:
cluster_ids = ["cluster"]
image_dir = Path("/home/ppzjbg/JGM_Tests/Mock_Images/")
outdir = Path("/home/ppztk1/Erosita/Outputs_Clusters/")
filters = ["VIS"]

process_cluster_pipeline_updated(
    image_dir=image_dir,
    outdir=outdir,
    cluster_ids=cluster_ids,
    filters=filters,
    mock_image=True,
    masking=False,
    rerun_masking = False,
    run_autoprof_function = True, 
    SB_extraction=True,
    show_profiles = True,
    prefix = 'no_noise' 
)

In [51]:

cluster_ids = ["cluster0"]
image_dir = Path("/home/ppzjbg/JGM_Tests/Mock_Images/")
outdir = Path("/home/ppztk1/Erosita/Outputs_Clusters/")
filters = ["H"]

process_cluster_pipeline_updated(
    image_dir=image_dir,
    outdir=outdir,
    cluster_ids=cluster_ids,
    filters=filters,
    mock_image=True,
    masking=True,
    rerun_masking=True,
    run_autoprof_function = True,
    SB_extraction=True,
    show_profiles = True,
    prefix='skypatch_field',
)



Using H band image and H band mask for cluster0
Processing /home/ppzjbg/JGM_Tests/Mock_Images/cluster0_H_skypatch_field.fits
Creating new NIR masks...
INFO: first HDU with data is extension 1. [astropy.nddata.ccddata]


2025-04-16 13:37:24 - nicl.mask.create_bcg_mask - INFO - Creating BCG mask
2025-04-16 13:37:28 - nicl.mask.create_icl_mask - INFO - Creating ICL mask
2025-04-16 13:37:28 - nicl.mask.create_icl_mask - DEBUG - Estimating background with box size 300 and filter size 3
2025-04-16 13:37:37 - nicl.mask.create_icl_mask - DEBUG - Average 0.5 sigma detection threshold: 0.01024
2025-04-16 13:37:37 - nicl.mask.create_icl_mask - DEBUG - Smoothing image with sigma=50.0
2025-04-16 13:37:37 - nicl.mask.create_icl_mask - DEBUG - Smoothing with a sampled median filter
2025-04-16 13:37:55 - nicl.mask.create_icl_mask - DEBUG - Detecting sources
2025-04-16 13:37:58 - nicl.mask.create_icl_mask - DEBUG - Keeping segment at specified position
2025-04-16 13:38:00 - nicl.mask.create_icl_mask - DEBUG - Creating source mask with dilation radius 70.0
2025-04-16 13:44:03 - nicl.mask.create_icl_mask - INFO - ICL mask created, 1353731 pixels masked (3.76%)
2025-04-16 13:44:03 - nicl.mask.create_object_mask - INFO - 

INFO: first HDU with data is extension 1. [astropy.nddata.ccddata]


2025-04-16 13:47:22 - nicl.mask.create_bcg_mask - INFO - Creating BCG mask
2025-04-16 13:47:25 - nicl.mask.create_icl_mask - INFO - Creating ICL mask
2025-04-16 13:47:25 - nicl.mask.create_icl_mask - DEBUG - Estimating background with box size 300 and filter size 3
2025-04-16 13:47:34 - nicl.mask.create_icl_mask - DEBUG - Average 0.5 sigma detection threshold: 0.00996
2025-04-16 13:47:34 - nicl.mask.create_icl_mask - DEBUG - Smoothing image with sigma=50.0
2025-04-16 13:47:34 - nicl.mask.create_icl_mask - DEBUG - Smoothing with a sampled median filter
2025-04-16 13:47:52 - nicl.mask.create_icl_mask - DEBUG - Detecting sources
2025-04-16 13:47:54 - nicl.mask.create_icl_mask - DEBUG - Keeping segment at specified position
2025-04-16 13:47:55 - nicl.mask.create_icl_mask - DEBUG - Creating source mask with dilation radius 70.0
2025-04-16 13:54:08 - nicl.mask.create_icl_mask - INFO - ICL mask created, 1032651 pixels masked (2.87%)
2025-04-16 13:54:08 - nicl.mask.create_object_mask - INFO - 

INFO: first HDU with data is extension 1. [astropy.nddata.ccddata]


2025-04-16 13:57:28 - nicl.mask.create_bcg_mask - INFO - Creating BCG mask
2025-04-16 13:57:33 - nicl.mask.create_icl_mask - INFO - Creating ICL mask
2025-04-16 13:57:33 - nicl.mask.create_icl_mask - DEBUG - Estimating background with box size 300 and filter size 3
2025-04-16 13:57:42 - nicl.mask.create_icl_mask - DEBUG - Average 0.5 sigma detection threshold: 0.01127
2025-04-16 13:57:42 - nicl.mask.create_icl_mask - DEBUG - Smoothing image with sigma=50.0
2025-04-16 13:57:42 - nicl.mask.create_icl_mask - DEBUG - Smoothing with a sampled median filter
2025-04-16 13:58:02 - nicl.mask.create_icl_mask - DEBUG - Detecting sources
2025-04-16 13:58:03 - nicl.mask.create_icl_mask - DEBUG - Keeping segment at specified position
2025-04-16 13:58:04 - nicl.mask.create_icl_mask - DEBUG - Creating source mask with dilation radius 70.0
2025-04-16 14:04:17 - nicl.mask.create_icl_mask - INFO - ICL mask created, 714684 pixels masked (1.99%)
2025-04-16 14:04:17 - nicl.mask.create_object_mask - INFO - C

Proceeding to stack_nir_bands...
INFO: using the unit adu passed to the FITS reader instead of the unit adu in the FITS file. [astropy.nddata.ccddata]
INFO: using the unit adu passed to the FITS reader instead of the unit adu in the FITS file. [astropy.nddata.ccddata]
INFO: using the unit adu passed to the FITS reader instead of the unit adu in the FITS file. [astropy.nddata.ccddata]
stack_nir_bands is complete...
Coadded image is saved.
Masking the coadded image...


KeyError: 'ZPAB'